# Tracking Spatial QA

This notebook is the maintained QA surface for tracking-backed spatial checks around throw time. It answers three operational questions:

- do shooter and goalkeeper positions exist at the refined throw timestamp
- which fixtures lose goalkeeper coverage or produce risky sync rows
- do the spatial distributions around the throw look plausible

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import PercentFormatter

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'data' / 'hbl_raw.duckdb').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root containing data/hbl_raw.duckdb')

REPO_ROOT = find_repo_root()
DB_PATH = REPO_ROOT / 'data' / 'hbl_raw.duckdb'
con = duckdb.connect(str(DB_PATH), read_only=True)
DB_PATH

In [ ]:
df_tracking = con.execute("""
WITH base AS (
    SELECT
        fixture_id,
        event_id,
        throw_timestamp_ms,
        person_league_id,
        goalkeeper_league_id,
        attack_type,
        sub_type,
        match_method,
        time_difference_ms,
        shot_position_x,
        shot_position_y,
        goal_position
    FROM shot_events
    WHERE throw_timestamp_ms IS NOT NULL
),
shooter AS (
    SELECT fixture_id, timestamp_ms, league_id, x_m AS shooter_x_m, y_m AS shooter_y_m
    FROM match_positions_normalized
),
goalkeeper AS (
    SELECT fixture_id, timestamp_ms, league_id, x_m AS goalkeeper_x_m, y_m AS goalkeeper_y_m
    FROM match_positions_normalized
)
SELECT
    b.fixture_id,
    b.event_id,
    b.throw_timestamp_ms,
    b.attack_type,
    b.sub_type,
    b.match_method,
    b.time_difference_ms,
    b.shot_position_x,
    b.shot_position_y,
    b.goal_position,
    s.shooter_x_m,
    s.shooter_y_m,
    g.goalkeeper_x_m,
    g.goalkeeper_y_m
FROM base AS b
LEFT JOIN shooter AS s
    ON s.fixture_id = b.fixture_id
   AND s.timestamp_ms = CAST(b.throw_timestamp_ms AS BIGINT)
   AND s.league_id = b.person_league_id
LEFT JOIN goalkeeper AS g
    ON g.fixture_id = b.fixture_id
   AND g.timestamp_ms = CAST(b.throw_timestamp_ms AS BIGINT)
   AND g.league_id = b.goalkeeper_league_id
""").fetchdf()

df_tracking['has_shooter_pos'] = df_tracking['shooter_x_m'].notna()
df_tracking['has_goalkeeper_pos'] = df_tracking['goalkeeper_x_m'].notna()
df_tracking['has_both_positions'] = df_tracking['has_shooter_pos'] & df_tracking['has_goalkeeper_pos']
df_tracking['time_difference_s'] = df_tracking['time_difference_ms'] / 1000.0

summary = pd.DataFrame({
    'rows_with_throw_timestamp': [len(df_tracking)],
    'fixtures': [df_tracking['fixture_id'].nunique()],
    'shooter_position_rows': [int(df_tracking['has_shooter_pos'].sum())],
    'goalkeeper_position_rows': [int(df_tracking['has_goalkeeper_pos'].sum())],
    'rows_with_both_positions': [int(df_tracking['has_both_positions'].sum())],
    'median_time_diff_ms': [float(df_tracking['time_difference_ms'].dropna().median())],
})
summary.T.rename(columns={0: 'value'})

In [ ]:
coverage = pd.DataFrame({
    'metric': ['shooter position', 'goalkeeper position', 'both positions'],
    'share': [
        df_tracking['has_shooter_pos'].mean(),
        df_tracking['has_goalkeeper_pos'].mean(),
        df_tracking['has_both_positions'].mean(),
    ],
})

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(coverage['metric'], coverage['share'], color=['#1f77b4', '#54a24b', '#f58518'])
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, 1.05)
ax.set_title('Coverage of tracking positions at refined throw timestamp')
ax.set_xlabel('Coverage metric')
ax.set_ylabel('Share of throw-timestamp rows')

for bar, share in zip(bars, coverage['share']):
    ax.text(bar.get_x() + bar.get_width() / 2, share + 0.015, f'{share:.1%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()
coverage

In [ ]:
fixture_risk = (
    df_tracking.groupby('fixture_id', dropna=False)
    .agg(
        rows=('event_id', 'count'),
        missing_goalkeeper_positions=('has_goalkeeper_pos', lambda values: int((~values).sum())),
        override_rows=('match_method', lambda values: int((values == 'time_priority_override').sum())),
        fallback_rows=('match_method', lambda values: int((values == 'time_only_fallback').sum())),
        median_time_diff_ms=('time_difference_ms', 'median'),
    )
    .reset_index()
)
fixture_risk['manual_review_share'] = (
    fixture_risk['missing_goalkeeper_positions'] + fixture_risk['override_rows'] + fixture_risk['fallback_rows']
 ) / fixture_risk['rows']
top_fixtures = fixture_risk.sort_values(['manual_review_share', 'rows'], ascending=[False, False]).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_fixtures['fixture_id'], top_fixtures['manual_review_share'], color='#d62728', alpha=0.85)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title('Fixtures with the highest spatial QA review share')
ax.set_xlabel('Missing goalkeeper positions + override + fallback share')
ax.set_ylabel('Fixture')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

top_fixtures

In [ ]:
spatial = df_tracking.loc[df_tracking['has_both_positions']].copy()
spatial['goalkeeper_depth_m'] = spatial['goalkeeper_x_m'].abs()
spatial['shooter_lateral_m'] = spatial['shooter_y_m']

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(
    spatial['shooter_x_m'],
    spatial['shooter_y_m'],
    c=spatial['goalkeeper_depth_m'],
    cmap=LinearSegmentedColormap.from_list('handball_depth', ['#cfe8f3', '#2c7fb8', '#08306b']),
    alpha=0.45,
    s=18,
    edgecolors='none',
)
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.axhline(0, color='black', linewidth=1, linestyle=':')
ax.set_title('Shooter positions at throw time, colored by goalkeeper depth')
ax.set_xlabel('Shooter x position (m)')
ax.set_ylabel('Shooter y position (m)')
fig.colorbar(scatter, ax=ax, label='Absolute goalkeeper x position (m)')
plt.tight_layout()
plt.show()

spatial[['fixture_id', 'event_id', 'match_method', 'shooter_x_m', 'shooter_y_m', 'goalkeeper_x_m', 'goalkeeper_y_m']].head(20)

In [ ]:
review_queue = (
    df_tracking.loc[
        (~df_tracking['has_goalkeeper_pos'])
        | df_tracking['match_method'].isin(['time_priority_override', 'time_only_fallback'])
    ]
    .sort_values(['has_goalkeeper_pos', 'time_difference_ms'], ascending=[True, False])
    [['fixture_id', 'event_id', 'match_method', 'attack_type', 'sub_type', 'time_difference_ms', 'has_shooter_pos', 'has_goalkeeper_pos']]
    .head(30)
)
review_queue